# 04 - Bronze : landing HDFS → table Delta bronze

Dans cette étape je lis les 522 CSV que j'ai déposés dans HDFS et je les écris dans une table Delta, sans les transformer.

- Je lis toutes les colonnes en texte  : comme ça aucune valeur n'est perdue ou mal convertie. Le typage, je le ferai en silver.
- J'ajoute juste 2 colonnes pour savoir d'où vient chaque ligne : date_ingestion et fichier_source.
- Je regroupe les 522 petits fichiers en 4 fichiers Parquet (problème des petits fichiers vu à l'étape 3).

## 1. J'ouvre la SparkSession

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = SparkSession.builder.appName("04_bronze").getOrCreate()

print("Application :", spark.sparkContext.appName)
print("Master      :", spark.sparkContext.master)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 14:21:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Application : 04_bronze
Master      : spark://spark-master:7077


## 2. Je lis les CSV dans HDFS (toutes les colonnes en texte)

In [2]:
SOURCE = "/landing/seismes/*/*.csv"   # dans HDFS : un dossier par année

brut = (
    spark.read
    .option("header", True)     # la 1re ligne contient les noms des colonnes
    .option("quote", '"')       # les textes avec des virgules sont entre guillemets
    .csv(SOURCE)
)

print("Nombre de colonnes lues :", len(brut.columns))
brut.printSchema()

Nombre de colonnes lues : 22
root
 |-- time: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- depth: string (nullable = true)
 |-- mag: string (nullable = true)
 |-- magType: string (nullable = true)
 |-- nst: string (nullable = true)
 |-- gap: string (nullable = true)
 |-- dmin: string (nullable = true)
 |-- rms: string (nullable = true)
 |-- net: string (nullable = true)
 |-- id: string (nullable = true)
 |-- updated: string (nullable = true)
 |-- place: string (nullable = true)
 |-- type: string (nullable = true)
 |-- horizontalError: string (nullable = true)
 |-- depthError: string (nullable = true)
 |-- magError: string (nullable = true)
 |-- magNst: string (nullable = true)
 |-- status: string (nullable = true)
 |-- locationSource: string (nullable = true)
 |-- magSource: string (nullable = true)



## 3. J'ajoute les colonnes de traçabilité
- date_ingestion: quand la ligne est arrivée dans bronze
- fichier_source : de quel CSV elle vient

In [3]:
bronze = (
    brut
    .withColumn("date_ingestion", F.current_timestamp())
    .withColumn("fichier_source", F.col("_metadata.file_path"))
)

bronze.select("id", "time", "mag", "type", "fichier_source").show(3, truncate=False)

+----------+------------------------+----+----------+----------------------------------------------------------------+
|id        |time                    |mag |type      |fichier_source                                                  |
+----------+------------------------+----+----------+----------------------------------------------------------------+
|ci38445863|2019-07-05T00:00:20.000Z|1.91|earthquake|hdfs://namenode:9000/landing/seismes/2019/seismes_2019-07-05.csv|
|ci37238596|2019-07-05T00:01:01.890Z|1.79|earthquake|hdfs://namenode:9000/landing/seismes/2019/seismes_2019-07-05.csv|
|ci37238636|2019-07-05T00:01:26.130Z|1.3 |earthquake|hdfs://namenode:9000/landing/seismes/2019/seismes_2019-07-05.csv|
+----------+------------------------+----+----------+----------------------------------------------------------------+
only showing top 3 rows



## 4. J'écris la table Delta bronze dans HDFS
- repartition(4) : je regroupe tout en 4 fichiers (au lieu de 522 petits).
- `ode("overwrite"): bronze est une copie exacte de la landing, donc si je relance la cellule, la table est remplacée et il n'y a pas de doublons.

In [4]:
CHEMIN_BRONZE = "/lakehouse/bronze/seismes"
NB_FICHIERS = 4

t0 = time.time()
(
    bronze.repartition(NB_FICHIERS)
    .write.format("delta")
    .mode("overwrite")
    .save(CHEMIN_BRONZE)
)
print(f"Table bronze écrite en {time.time() - t0:.0f} s")

Table bronze écrite en 93 s


## 5. Je relis la table et je vérifie qu'il ne manque aucune ligne


In [5]:
table = spark.read.format("delta").load(CHEMIN_BRONZE).cache()

nb_lignes = table.count()
nb_fichiers_sources = table.select("fichier_source").distinct().count()

print("Lignes dans bronze :", nb_lignes, "(attendu : même total que le notebook 02)")
print("Fichiers sources   :", nb_fichiers_sources, "(attendu : 522)")
print("Colonnes           :", len(table.columns), "(22 colonnes USGS + 2 de traçabilité = 24)")

26/11/12 14:26:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Lignes dans bronze : 1604291 (attendu : même total que le notebook 02)
Fichiers sources   : 522 (attendu : 522)
Colonnes           : 24 (22 colonnes USGS + 2 de traçabilité = 24)


## 6. Je vérifie que les colonnes ne sont pas décalées

In [6]:
(
    table.groupBy("type")
    .count()
    .withColumnRenamed("count", "nb_evenements")
    .withColumn("pourcentage", F.round(F.col("nb_evenements") * 100 / nb_lignes, 2))
    .orderBy(F.desc("nb_evenements"))
    .show(truncate=False)
)

+--------------------------+-------------+-----------+
|type                      |nb_evenements|pourcentage|
+--------------------------+-------------+-----------+
|earthquake                |1567121      |97.68      |
|quarry blast              |13308        |0.83       |
|ice quake                 |9963         |0.62       |
|explosion                 |9898         |0.62       |
|mining explosion          |2262         |0.14       |
|other event               |1405         |0.09       |
|chemical explosion        |131          |0.01       |
|volcanic eruption         |74           |0.0        |
|landslide                 |32           |0.0        |
|sonic boom                |31           |0.0        |
|mine collapse             |19           |0.0        |
|rock burst                |12           |0.0        |
|experimental explosion    |11           |0.0        |
|not reported              |11           |0.0        |
|nuclear explosion         |3            |0.0        |
|Landslide

## 7. Premières observations sur les données brutes
Je regarde ce que je devrai gérer en silver : des identifiants en double ? des magnitudes vides ?

In [7]:
nb_ids_distincts = table.select("id").distinct().count()
nb_mag_vides = table.filter(F.col("mag").isNull() | (F.trim(F.col("mag")) == "")).count()
nb_seismes = table.filter(F.col("type") == "earthquake").count()

print("Identifiants en double :", nb_lignes - nb_ids_distincts)
print("Magnitudes vides       :", nb_mag_vides)
print(f"Séismes (earthquake)   : {nb_seismes} soit {nb_seismes * 100 / nb_lignes:.1f} %")

Identifiants en double : 0
Magnitudes vides       : 626
Séismes (earthquake)   : 1567121 soit 97.7 %


## 8. Je vérifie que le problème des petits fichiers est résolu


In [8]:
details = DeltaTable.forPath(spark, CHEMIN_BRONZE).detail().select("numFiles", "sizeInBytes").first()

print("Avant (landing) : 522 fichiers CSV, environ 300 Mo")
print(f"Après (bronze)  : {details['numFiles']} fichiers Parquet, {details['sizeInBytes'] / 1e6:.0f} Mo")
print(f"Taille moyenne d'un fichier : {details['sizeInBytes'] / details['numFiles'] / 1e6:.0f} Mo")

Avant (landing) : 522 fichiers CSV, environ 300 Mo
Après (bronze)  : 4 fichiers Parquet, 131 Mo
Taille moyenne d'un fichier : 33 Mo


## 9. L'historique de la table Delta



In [9]:
(
    DeltaTable.forPath(spark, CHEMIN_BRONZE)
    .history()
    .select("version", "timestamp", "operation", "operationMetrics.numOutputRows")
    .orderBy("version")
    .show(truncate=False)
)

+-------+-----------------------+---------+-------------+
|version|timestamp              |operation|numOutputRows|
+-------+-----------------------+---------+-------------+
|0      |2026-11-12 14:25:30.666|WRITE    |1604291      |
+-------+-----------------------+---------+-------------+



## 10. Je libère le cache et je ferme la SparkSession (pour libérer les cœurs du cluster)

In [10]:
table.unpersist()
spark.stop()